# 🚀 Distill a tool-use agent using Knowledge Distillation on Amazon SageMaker AI

Sequence-level knowledge distillation (SeqKD) of a shopping **tool-use agent** into `Qwen/Qwen3.5-4B`,
trained with the **serverless `SFTTrainer`** from the SageMaker Python SDK v3.

SeqKD needs no distillation-specific algorithm: a larger teacher generates tool-use trajectories, and the
student is trained on them with ordinary SFT (loss masked to the completion). The teacher trajectories
persist as a dataset, so you can reuse them to train other students.

**What you will learn**

1. Encode teacher ShoppingBench trajectories into `{prompt, completion}` pairs with the student's exact
   non-thinking chat template.
2. Run SeqKD (SFT on those pairs) as a SageMaker serverless training job.
3. Deploy the customized student and evaluate task success (ASR) + IFEval regression.

**Why the serverless `SFTTrainer`:** SageMaker runs a managed recipe, so you supply a dataset and
hyperparameters and nothing else. You don't choose an instance type, build a training image, or write a
training script — and, specifically for this model, you don't hand-patch kernels. Qwen3.5-4B is a
linear-attention/conv1d **hybrid** whose linear-attn path needs `flash-linear-attention` force-installed
against a conflicting `triton` pin, plus `sdpa` attention to avoid a CUDA illegal-memory-access on the
managed driver. The recipe handles that. The job also emits an **already-merged, registered Model
Package**, which you deploy and evaluate directly — no adapter-merge step.

**Results with this notebook.** ShoppingBench's 250-case product test split overlaps the public
teacher-trajectory dataset, so the headline ASR is measured on the **decontaminated 103-case subset** —
the eval cases whose query does *not* appear anywhere in training (step 01 builds that split for you):

| Metric | Base Qwen3.5-4B | Distilled student | Δ |
|---|---|---|---|
| Tool-use **ASR**, decontaminated 103 cases | 0.379 | **0.495** | **+11.6 pts** |
| `recommend_product` completion rate | 0.650 | **0.864** | +21.4 pts |
| **IFEval** (prompt-level strict, non-thinking) | 0.826 | 0.782 | −4.4 pts |
| **MMLU** (5-shot, 57 subtasks) | 0.697 | 0.700 | +0.3 pts |

Agent skill rises substantially while general instruction-following and knowledge hold — the trade you
want from SeqKD.

**And *why* it rises.** A per-trajectory LLM reasoning judge (04c) scores each rollout's reasoning
quality alongside its outcome, pairwise and blind over the same 103 clean cases. The gain is
**process discipline, not judgement**: `efficiency` +1.18 (p<0.001) and `search_strategy` +0.37
(p=0.037) move, while `verification` (+0.10) and `constraint_tracking` (+0.09) do not. The base model
never issues `recommend_product` at all on 36 of 103 cases, and on exactly those `efficiency` goes
1.89 → 3.86. SeqKD taught the agent to **finish the rollout**. That also explains what still fails —
`orm.py`'s `sku & attrs` axis needs exact `(key, value)` matches, which is a verification problem, and
verification is the one axis distillation did not improve.

> **Why 103 and not 250 — read this before quoting any ShoppingBench number.** The public teacher set
> `oro-ai/sn15-shoppingbench-sft-15k` has only **469 unique queries**, and **147 of the 250 eval queries
> appear in it verbatim**. For **110** of those, a training trajectory recommends the eval case's exact
> gold `product_id` — so 44% of the benchmark ships its own answer key inside the training data.
>
> The cost of ignoring this is large and measurable. Scoring the identical rollouts three ways:
>
> | split | base | student | lift |
> |---|---|---|---|
> | all 250 | 0.432 | 0.636 | +20.4 |
> | **clean 103** | **0.379** | **0.495** | **+11.6** |
> | contaminated 147 | 0.469 | 0.735 | +26.6 |
>
> Contamination nearly **doubles** the apparent lift. The contaminated subset is also intrinsically
> easier — the base, which never saw training data, gains +9.0 pts on it — so about **15 points** of the
> student's advantage there is memorization. `+11.6` is the number that survives scrutiny.
>
> This is a property of the public dataset, not of SeqKD or of SageMaker. Any benchmark whose training
> and test instructions come from one generated pool deserves the same check before you trust a lift.
>
> **Provenance of the numbers above, stated exactly:** they come from a student trained on the *full*
> 58,298-example file and scored on the clean 103. That score is sound — no eval answer leaked into the
> metric — but step 01 of this notebook goes further and also *removes* the 5,679 contaminated
> trajectories from training, leaving 38,535 examples (66%). Training on a third less data will shift
> the result somewhat, so treat `+11.6` as the reference for the lift's shape and magnitude, not as a
> figure to reproduce to three decimals.

## 00. Setup

The serverless path lives in SDK **v3** namespaces: `sagemaker.train.sft_trainer.SFTTrainer`,
`sagemaker.ai_registry.dataset.DataSet`, and `sagemaker.core.resources`.

In [ ]:
import os, json, boto3
from sagemaker.core.helper.session_helper import get_execution_role

REGION = boto3.Session().region_name or "us-west-2"
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = REGION

# Needed for native MLflow metrics while the trainer waits:
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{REGION}.app.aws"

try:
    ROLE = get_execution_role()
    print("Execution role:", ROLE)
except Exception:
    ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
    ROLE = f"arn:aws:iam::{ACCOUNT}:role/Dev"        # <- replace with your execution role

BUCKET  = "<name>"                                   # <- replace with your bucket
PREFIX  = "<name>"
S3_ROOT = f"s3://{BUCKET}/{PREFIX}"
print(REGION, ROLE, S3_ROOT)

## 01. Data Prep — encode teacher trajectories → `{prompt, completion}`

One training example per assistant step; loss falls only on the completion. We render with the **student's own chat template** (`apply_chat_template`, `enable_thinking=False`, `tools=`) via a prefix-diff so `prompt + completion` is byte-identical to inference (non-thinking: prompt ends with a closed-empty `<think></think>`; tool calls render in Qwen-native format). TRL auto-detects the prompt/completion format and masks the prompt. Swap `REPO`/`sft_file` for your own teacher trajectories.

In [ ]:
import json, random, urllib.request
from huggingface_hub import list_repo_files, hf_hub_download
from transformers import AutoTokenizer
HF_STUDENT="Qwen/Qwen3.5-4B"
REPO = "oro-ai/sn15-shoppingbench-sft-15k"   # example teacher trajectories
jsonls = [f for f in list_repo_files(REPO, repo_type="dataset") if f.endswith(".jsonl")]
sft_file = next(f for f in jsonls if "metadata" not in f.lower() and "eval" not in f.lower())
raw = [json.loads(l) for l in open(hf_hub_download(REPO, sft_file, repo_type="dataset")) if l.strip()]
raw = [r for r in raw if r.get("messages")]
print(len(raw), "trajectories")

# --- DECONTAMINATION (do not skip) -------------------------------------------------------------
# This teacher set draws on only ~469 unique queries, and 147 of ShoppingBench's 250 product-test
# queries appear in it verbatim -- 110 of them alongside the eval case's exact gold product_id. Train
# on the raw file and the eval set is scoring recall, not skill (measured: +20.4 ASR vs +11.6 clean).
SB_TEST = ("https://raw.githubusercontent.com/yjwjy/ShoppingBench/main/"
           "data/synthesize_product_test.jsonl")
eval_rows = [json.loads(l) for l in urllib.request.urlopen(SB_TEST).read().decode().splitlines() if l.strip()]
eval_queries = {r["query"] for r in eval_rows}

def traj_query(t):                      # a trajectory's task == its first user message
    return next((m["content"] for m in t["messages"] if m.get("role") == "user"), None)

train_queries = {traj_query(t) for t in raw}
overlap = eval_queries & train_queries
print(f"{len(train_queries)} unique train queries, {len(eval_queries)} eval queries, "
      f"{len(overlap)} overlap ({len(overlap)/len(eval_queries):.1%} of the eval set)")

# Split the benchmark by query. Train only on trajectories that touch no held-out query...
before = len(raw)
raw = [t for t in raw if traj_query(t) not in eval_queries]
print(f"train: {before} -> {len(raw)} trajectories ({before - len(raw)} dropped as contaminated)")

# ...and score only on eval cases the ORIGINAL training file never saw. Step 04 points
# run_evaluate.py at this file instead of the shipped 250-case one.
clean_rows = [r for r in eval_rows if r["query"] not in overlap]
with open("synthesize_product_test_clean.jsonl", "w") as f:
    for r in clean_rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"eval: {len(clean_rows)} / {len(eval_rows)} cases clean "
      f"-> synthesize_product_test_clean.jsonl")
# -----------------------------------------------------------------------------------------------

tok = AutoTokenizer.from_pretrained(HF_STUDENT, trust_remote_code=True)
SYSTEM = ("You are a shopping assistant. Use the available tools to satisfy the request: "
          "search the catalog, inspect candidates, verify the constraints, then recommend.")

def _render(msgs, tools, add_gen):
    return tok.apply_chat_template(msgs, tools=tools, tokenize=False,
                                   add_generation_prompt=add_gen, enable_thinking=False)

def to_rows(traj):
    tools = traj.get("tools")
    convo = [{"role": "system", "content": SYSTEM}] + [m for m in traj["messages"] if m.get("role") != "system"]
    rows = []
    for i, m in enumerate(convo):
        if m.get("role") != "assistant":
            continue
        prompt = _render(convo[:i],     tools, add_gen=True)
        full   = _render(convo[:i + 1], tools, add_gen=False)
        assert full.startswith(prompt), "chat-template prefix mismatch"
        completion = full[len(prompt):]
        if completion.strip():
            rows.append({"prompt": prompt, "completion": completion})
    return rows

examples = [r for traj in raw for r in to_rows(traj)]
random.seed(42); random.shuffle(examples)
val, train = examples[:1000], examples[1000:]
print(f"{len(raw)} trajectories -> {len(examples)} SFT examples  (train {len(train)} / val {len(val)})")

def write_jsonl(rows, p):
    with open(p, "w") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
write_jsonl(train, "seqkd_train.jsonl"); write_jsonl(val, "seqkd_val.jsonl")

#### Upload to S3, then register the dataset in the SageMaker AI Registry

The serverless `SFTTrainer` consumes a **registered dataset ARN**, not a raw S3 path. `DataSet.create`
wraps a JSONL file in a versioned registry entry. Each line must carry `prompt` / `completion` fields —
exactly what the cell above wrote.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

s3 = boto3.client("s3", region_name=REGION)
s3.upload_file("seqkd_train.jsonl", BUCKET, f"{PREFIX}/data/seqkd_train.jsonl")
s3.upload_file("seqkd_val.jsonl",   BUCKET, f"{PREFIX}/data/seqkd_val.jsonl")
TRAIN_S3 = f"{S3_ROOT}/data/seqkd_train.jsonl"
VAL_S3   = f"{S3_ROOT}/data/seqkd_val.jsonl"

train_dataset = DataSet.create(name="seqkd-shopping-train", source=TRAIN_S3)
val_dataset   = DataSet.create(name="seqkd-shopping-val",   source=VAL_S3)
print("train:", train_dataset.arn)
print("val:  ", val_dataset.arn)

## 02. Fine-tune (SeqKD) with serverless `SFTTrainer`

Distillation here *is* ordinary SFT on the teacher trajectories. You give the trainer a base model id, a
training type, the two registered datasets, and an output path.

`model_package_group` is **required** when you pass a base model id — it's where the output Model Package
lands. `accept_eula=True` is required for gated base models.

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.core.resources import ModelPackageGroup

MPG_NAME = "seqkd-qwen35-4b-shop"
try:
    model_package_group = ModelPackageGroup.create(model_package_group_name=MPG_NAME)
except Exception as e:
    print("reusing existing group:", type(e).__name__)
    model_package_group = MPG_NAME          # an existing name or ARN is also accepted

# To find out which models are supported by serverless training:
# SFTTrainer.list_supported_models()

sft_trainer = SFTTrainer(
    model="huggingface-vlm-qwen3-5-4b",     # the Qwen3.5-4B student
    training_type=TrainingType.FULL,        # TrainingType.FULL for FFT -- read the note below first
    model_package_group=model_package_group,
    mlflow_experiment_name="seqkd-shopping-exp",
    mlflow_run_name="seqkd-shopping",
    training_dataset=train_dataset.arn,
    validation_dataset=val_dataset.arn,
    s3_output_path=f"{S3_ROOT}/output/",
    accept_eula=True,
)

### Inspect, then override the hyperparameters if needed

Print the recipe defaults first and change only what you need.

In [ ]:
print("Recipe defaults:")
pprint(sft_trainer.hyperparameters.to_dict())

In [ ]:
sft_trainer.hyperparameters.dataset_max_len       = 16384
sft_trainer.hyperparameters.max_epochs            = 2
sft_trainer.hyperparameters.global_batch_size     = 64
sft_trainer.hyperparameters.learning_rate         = 0.00002
sft_trainer.hyperparameters.train_val_split_ratio = 0.9
sft_trainer.hyperparameters.lr_warmup_steps_ratio = 0.03
pprint(sft_trainer.hyperparameters.to_dict())

### Dry-run, then train

`dry_run=True` validates IAM permissions, dataset ARNs, hyperparameters, and infrastructure without
submitting a job — cheap insurance before a multi-hour run.

In [ ]:
sft_trainer.train(dry_run=True)
print("Dry run passed - configuration is valid.")

In [ ]:
training_job = sft_trainer.train(wait=True)

# To submit now and monitor later instead:
#   training_job = sft_trainer.train(wait=False)
#   sft_trainer.stream_logs()

print(training_job.training_job_name, training_job.training_job_status)

> **Hyperparameter notes, from the runs behind the table at the top.**
>
> - **`dataset_max_len = 16384`.** Multi-turn ReAct trajectories accumulate tool output. At 4096 roughly
>   19% of this dataset's trajectories get truncated mid-rollout, and ASR pays for it.
> - **`learning_rate = 5e-05` at `global_batch_size = 64`.** If the recipe pins you to a large global
>   batch, scale the LR to it rather than fighting it. Scale by `sqrt` if you change the batch size.
> - **FFT arm (`TrainingType.FULL`):** drop LR to **`2e-05`** and set `weight_decay = 0.0`. FFT reaches a
>   higher ASR ceiling (0.736 measured) but on shop-only data it catastrophically forgets — IFEval
>   collapsed to 0.155. Mixing roughly **30% general instruction data** back into the training file
>   recovered it (IFEval 0.702). Don't run pure FFT on agent-only data unless you accept that trade.
>   Data mixing matters more here than any LR choice.
> - LoRA at `r=16, alpha=128` was sufficient; raising the rank was not what moved ASR. Note that LoRA
>   scaling is `alpha / rank`, so going `r=16 → r=64` at a fixed `alpha=128` *weakens* the update 4× — it
>   is not a pure capacity increase. To isolate capacity at `r=64`, raise `alpha` to 512.
> - **Step count scales with your dataset.** The decontaminated file is **38,535 examples** (9,590 of
>   15,269 trajectories survive the eval-query filter — 66%); after the 1,000-row val holdout and
>   `train_val_split_ratio = 0.9`, ~33.8k training rows. At `global_batch_size = 64` that is ~528
>   optimizer steps per epoch (~1,056 for 2 epochs). A warmup ratio of `0.03`–`0.05` is appropriate at
>   that length; a few-hundred-example run would need proportionally less.

## 03. Collect the output Model Package

**There is no merge step here, and that is the point.** A self-managed LoRA job writes an *adapter* that
you then have to download, merge into the base weights, re-add the VLM processor configs
(`preprocessor_config.json`, `processor_config.json`), and re-upload. The serverless job registers a
ready-to-deploy **Model Package** instead; the completed job's `output_model_package_arn` points at it.

In [ ]:
job = TrainingJob.get(training_job_name=training_job.training_job_name)
MODEL_PKG_ARN = job.output_model_package_arn
print("Output Model Package ARN:", MODEL_PKG_ARN)

model_package = ModelPackage.get(model_package_name=MODEL_PKG_ARN)
print("status:", model_package.model_package_status)

# To find the package later without the job object:
#   aws sagemaker list-model-packages --model-package-group-name seqkd-qwen35-4b-shop \
#     --sort-by CreationTime --sort-order Descending \
#     --query 'ModelPackageSummaryList[0].ModelPackageArn'

## 04. Evaluate

| Axis | Compute | How |
|---|---|---|
| **MMLU** | SageMaker **managed, serverless** | `BenchMarkEvaluator` at the Model Package (loglikelihood, so no thinking artifact) |
| **IFEval + tool-use ASR** | a **real-time endpoint** (GPU) + a CPU-light driver here | deploy once, drive it through a LiteLLM shim (04b) |
| **Reasoning quality** | **no GPU** — Bedrock judge over 04b's saved rollouts | pairwise blind LLM judge, five process axes (04c) |

ASR is the headline metric; it needs the benchmark's Lucene index and search server, so it is a GPU-host
job rather than a notebook one. Run **04b first**, then **04c**, which re-reads 04b's rollout files and
needs no further inference — it is what turns "the number went up" into "here is which part of the
agent's process improved."

> **A dead end worth not repeating.** An earlier version of 04c used the managed `LLMAsJudgeEvaluator` to
> grade only the agent's *first* turn, which needs no rollouts at all. It cannot show the lift: first-turn
> tool choice is already at ceiling for the base model (measured 0.90/1.0), so the delta is flat. Two
> further traps if you try it — with no reference answers the builtin `Correctness` metric judges against
> a placeholder `referenceResponse` of `"dummy"`, and the managed inference path returns the completion
> wrapped in a Python `str(list)` repr with the opening `<think>` absorbed into the prompt, which makes any
> strict format metric score 0 for *every* model. Judge whole trajectories instead.

Generation runs on the endpoint's GPU; the rollout loop and scoring run in this notebook kernel. The
companion **`evaluate-tool-call-accuracy.ipynb`** does the full base-vs-student comparison and chart.

> **Measurement trap: IFEval must be served thinking-OFF.** Qwen3.5 defaults to thinking ON, and its chat
> template then opens a reasoning block — so IFEval scores the *reasoning trace* against the prompt's
> format constraints ("≤100 words", "wrap the response in quotes", "highlight 2 sections") and they fail by
> construction. Same weights, same harness: **0.782 non-thinking vs 0.264 thinking-ON.** The 0.264 is an
> artifact of the serving config, not a property of the model. An endpoint behind `/invocations` can't force
> `enable_thinking=false` through the bridge, so the trustworthy IFEval number comes from a standalone vLLM
> with a non-thinking chat template — see `scripts/eval_lmeval.sh`.

> **Self-contained alternative (no endpoint):** `scripts/eval_lmeval.sh <MODEL_DIR> <IFACE> <GPU> <TAG>`
> runs non-thinking IFEval on a single GPU box, and `scripts/shopping_asr_eval.sh` runs ASR.

### 04a. Deploy the student to a real-time endpoint (`ModelBuilder`)

Deploy straight from the Model Package produced in step 03 — no S3 model artifact plumbing. Name the LMI
image and `model_server=DJL_SERVING` **explicitly**: that image bundles **vLLM 0.20.1**, which already has
native kernels for the Qwen3.5 linear-attn/conv1d hybrid, so **no custom inference image is needed**. The
`triton` / `flash-linear-attention` workarounds are training-side only.

In [ ]:
from sagemaker.serve import ModelBuilder, ModelServer

ENDPOINT_NAME = "seqkd-qwen35-4b-shop"
INSTANCE_TYPE = "ml.g5.2xlarge"          # ml.p4de.24xlarge / ml.p4d.24xlarge also validated
LMI_IMAGE = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/djl-inference:0.36.0-lmi25.0.0-cu130"  # vLLM 0.20.1

mb = ModelBuilder(
    model=MODEL_PKG_ARN,                 # the registered output of the serverless job
    role_arn=ROLE,
    image_uri=LMI_IMAGE,
    model_server=ModelServer.DJL_SERVING,
    instance_type=INSTANCE_TYPE,
    env_vars={"OPTION_ROLLING_BATCH": "vllm", "OPTION_TRUST_REMOTE_CODE": "true",
              "OPTION_TENSOR_PARALLEL_DEGREE": "1", "OPTION_MAX_MODEL_LEN": "8192"},
)
mb.model_name = ENDPOINT_NAME
mb.build()
mb.deploy(endpoint_name=ENDPOINT_NAME, initial_instance_count=1, instance_type=INSTANCE_TYPE)
print("endpoint:", ENDPOINT_NAME)

# Capacity note: g5/g6 2xlarge often return InsufficientInstanceCapacity. A Failed endpoint must be
# deleted before you retry the same name, and SageMaker won't delete one stuck in Creating until it settles.

### 04b. IFEval + tool-use ASR — notebook driver → the endpoint

These run in **this notebook kernel** (CPU-light — orchestration + scoring only) and send requests to the endpoint, whose GPU does the generation. SageMaker endpoints are invoked via `InvokeEndpoint`, not an OpenAI URL, so **LiteLLM** bridges the gap — either call `litellm.completion(model=f"sagemaker/{ENDPOINT_NAME}", ...)` directly, or run `litellm --model sagemaker/<endpoint>` as a proxy that exposes an OpenAI `base_url` for `lm_eval` / the rollout to point at.

**Thinking-off (important):** on a managed endpoint you can't pass `--chat-template nonthink_template.jinja` like the self-hosted path, so force non-thinking **per request** with `extra_body={"enable_thinking": false}` (mirrors the rollout config). IFEval floors (~28%) if thinking is left on.

**ASR** reuses the same ReAct rollout as `scripts/shopping_asr_eval.sh` / `run_rollout.py`, only with `base_url` pointed at the LiteLLM shim instead of a local vLLM. It still needs the **ShoppingBench repo + search server (:5631) + `JAVA_HOME`** for tool results and orm.py scoring.

**You must write the rollout config yourself.** ShoppingBench drives `run_rollout.py` from a per-model JSON under `config/rollout/`, and upstream ships one for `gpt-4o`, `qwen3-*`, `gemma-*` and friends — but **none for Qwen3.5-4B**. The next cell writes one for the base model and one for the student. Keep every field except the three marked ones identical to the upstream configs: the scoring path has to stay byte-identical to the published numbers for the comparison to mean anything.

In [ ]:
# ShoppingBench has no Qwen3.5-4B rollout config — write one per model you want to score.
# Only the three fields marked <- differ between runs; everything else mirrors the upstream configs.
import json
from pathlib import Path

SB = Path("/path/to/ShoppingBench")        # the cloned harness (src/agent/run_rollout.py lives here)
PORT_BASE, PORT_STUDENT = 30001, 30066     # one vLLM per GPU -> a distinct port per model


def write_rollout_config(tag, served_name, port):
    cfg = {
        "task": "product",
        "system_prompt_file": "src/agent/prompt/rollout.md",
        "synthesize_file": "data/synthesize_product_test.jsonl",   # the 250-case held-out eval set
        "rollout_file": f"data/rollout_product_{tag}.jsonl",       # <- per-run trajectory output
        "threads": 4,
        "base_url": f"http://127.0.0.1:{port}/v1",                 # <- local vLLM, or the LiteLLM shim
        "api_key": "None",
        "model_config": {
            "model": served_name,              # <- MUST equal vllm --served-model-name, or you get
                                               #    empty completions and a low score with NO error
            "temperature": 0,
            "max_tokens": 8192,
            "stream": True,
            "extra_body": {"enable_thinking": False},   # Qwen3.5 thinks by default; the measured
                                                        # numbers are all thinking-off
        },
    }
    path = SB / "config" / "rollout" / f"{tag}_qwen35-4b.json"
    path.write_text(json.dumps(cfg, indent=2))
    print(path, "->", served_name, cfg["base_url"])
    return path


BASE_CFG    = write_rollout_config("base",  "qwen35-4b-base",  PORT_BASE)
STUDENT_CFG = write_rollout_config("seqkd", "qwen35-4b-seqkd", PORT_STUDENT)

# Then, per model:
#   bash scripts/shopping_asr_eval.sh <MODEL_DIR> <served_name> <cfg_path> <GPU> <PORT>
# ASR is the `success rate` line from run_evaluate.py (orm.py rule>=1). Do NOT report `gt rate` --
# that is strict exact product_id match and runs ~25 points lower on the same rollout.

In [ ]:
# --- VALIDATED 2026-10-01 against a live p4de endpoint (then torn down) ---
# IFEval over the endpoint works via a LiteLLM proxy + lm-eval, with these exact flags:
#   litellm --model sagemaker/<ENDPOINT_NAME> --port 4000 --drop_params      # --drop_params: sagemaker
#     provider rejects lm-eval's `seed` param (400) without it
#   lm_eval --model local-chat-completions \
#     --model_args base_url=http://localhost:4000/v1/chat/completions,model=sagemaker/<ENDPOINT_NAME>,\
#                  num_concurrent=4,tokenizer_backend=huggingface,tokenizer=Qwen/Qwen3.5-4B,eos_string=<|im_end|> \
#     --tasks ifeval --apply_chat_template --gen_kwargs max_tokens=1280
# CAVEAT (important): the DJL/LMI + LiteLLM bridge does NOT propagate enable_thinking=false (and --drop_params
# strips it), so the endpoint serves THINKING-ON -> IFEval is floored (sampled: prompt-strict ~0.18, the
# <think> block eats the token budget). To get the real non-thinking 0.826 you must either serve with the
# non-thinking template (pod: `vllm serve --chat-template nonthink_template.jinja`) or deploy with
# model_server=ModelServer.VLLM (vLLM-native OpenAI server, which honors chat_template_kwargs).

# Invoke schema: this endpoint serves the TGI `{"inputs":..., "parameters":...}` schema (returns {"generated_text"}).
# The OpenAI `{"messages":...}` schema returns 424 unless the chat route is enabled; LiteLLM's `sagemaker/` provider uses the TGI schema under the hood.
# --- LiteLLM proxy: expose the endpoint as an OpenAI base_url ---
#   pip install "litellm[proxy]"
#   litellm --model sagemaker/{ENDPOINT_NAME} --port 4000 &        # -> http://localhost:4000
#
# IFEval (non-thinking, generative) — point lm_eval at the proxy:
#   lm_eval --model local-chat-completions \
#     --model_args base_url=http://localhost:4000/v1/chat/completions,model=sagemaker/{ENDPOINT_NAME},num_concurrent=8 \
#     --apply_chat_template --gen_kwargs enable_thinking=false --tasks ifeval
#   # reference: prompt-strict 0.826 (base) -> 0.782 (student)

# ASR — drive the ShoppingBench ReAct loop against the endpoint (thinking OFF), score with orm.py:
import litellm
def endpoint_chat(messages, tools=None):
    return litellm.completion(model=f"sagemaker/{ENDPOINT_NAME}", messages=messages, tools=tools,
                              temperature=0, extra_body={"enable_thinking": False})
# for ex in eval_dataset: run the tool loop with endpoint_chat() (search server for tool results),
#     then score the final recommendation with orm.py (success rate = rule>=1)
# Score against the DECONTAMINATED split written in step 01, not the shipped 250-case file:
#   cfg["synthesize_file"] = "data/synthesize_product_test_clean.jsonl"
# run_evaluate.py keys on extra_info.query and skips queries absent from synthesize_rewards, with
# denominators len(results) -- so swapping synthesize_file re-scores the SAME rollout on the subset.
# No re-rollout and no GPU needed to go from the 250-case number to the clean one.
print("ASR reference (success rate): clean 103 cases, base 0.379 -> student 0.495  (+11.6 pts)")
print("  all 250 cases (contaminated, for contrast):  base 0.432 -> student 0.636  (+20.4 pts)")

### 04c. Per-trajectory reasoning judge — *why* ASR moves

ASR tells you **whether** the agent succeeded. It does not tell you **what changed in the agent's
process**, and that is the part a distillation write-up has to defend. So score each trajectory's
**reasoning quality alongside its outcome**, with an LLM judge over the *whole* rollout.

This step consumes the rollout files 04b already wrote — so it needs **no GPU and no new inference**,
but it does depend on 04b having run. (An earlier design judged only the agent's *first* turn via the
managed `LLMAsJudgeEvaluator`. Don't do that: first-turn tool choice is already at ceiling for the base
model — measured 0.90/1.0 — so it has no headroom and shows a flat delta. The lift is multi-turn.)

**Design choices that make the number defensible:**

- **Pairwise, not pointwise.** Base and student trajectories for the same query are judged side by side,
  so the judge calibrates against a concrete alternative instead of an absolute scale it has to invent.
- **Blind and order-randomized.** The judge is never told which arm is which, and a per-query seeded coin
  flip decides display order, which controls position bias. Verify the balance in the output.
- **Outcome withheld.** The judge does not see the ASR result, so reasoning scores cannot be
  rationalizations of a known winner. The outcome is joined afterwards — which lets you *validate the
  judge*: on cases where the two arms' ASR actually differs, the judge should prefer the arm that won.
- **Five process axes, scored separately**, because they do not move together and an average hides that:
  `constraint_tracking`, `search_strategy`, `evidence_grounding`, `verification`, `efficiency`.
- **Decontaminated 103 queries** from step 01, so this is not inflated by the train/eval overlap.

Reasoning lives in each step's `completion.content` *before* the closing `</think>` — the serving path
leaves `reasoning_content` empty and the opening `<think>` is template-supplied, so split on `</think>`
rather than looking for a matched pair.

**Measured result** (103 pairs, judge `claude-sonnet-4-5`, paired sign test):

| Reasoning axis | Base | Student | Δ | |
|---|---|---|---|---|
| `efficiency` | 2.68 | 3.86 | **+1.18** | p<0.001, W75/L21 |
| `search_strategy` | 3.50 | 3.87 | **+0.37** | p=0.037, W34/L18 |
| `evidence_grounding` | 3.13 | 3.41 | +0.28 | n.s. |
| `verification` | 2.78 | 2.87 | +0.10 | n.s. |
| `constraint_tracking` | 3.96 | 4.05 | +0.09 | n.s. |
| `overall_reasoning` | 3.16 | 3.56 | +0.39 | p=0.093, n=97 |

Head-to-head the student is preferred in **54 of 89 decided pairs (60.7%)**, p=0.056.

**Read this honestly: the gain is process discipline, not judgement.** `efficiency` and `search_strategy`
move; `verification` and `constraint_tracking` do not. The mechanism is visible in the data — the base
model never issues `recommend_product` at all on **36 of 103** cases, and on exactly those cases
`efficiency` goes 1.89 → 3.86 (**+1.97**) versus +0.76 where the base did recommend. SeqKD taught the
agent to *finish the rollout*, which is the same thing the raw completion rate says (0.650 → 0.864).

That also explains the residual ASR failures rather than hand-waving them: what still fails is the
`sku & attrs` axis of `orm.py`, which demands exact `(key, value)` tuple matches — a verification problem,
and `verification` is precisely the axis distillation did **not** improve.

**The judge is validated, not assumed.** On the pairs where the two arms' ASR differs and the judge is not
tied, it prefers the arm that actually succeeded in **19 of 22 cases (86.4%)**. Pooled across both arms,
mean `overall_reasoning` is 3.81 on succeeded trajectories versus 3.00 on failed.

In [ ]:
# Per-trajectory reasoning judge over the rollouts 04b produced. No GPU, no new inference.
# Rollout schema: each line is a list of steps; step["completion"]["content"] holds the think text
# before </think>; step["completion"]["message"] holds {tool_call, obs, response}.
import json, random, re, statistics as st, threading
from concurrent.futures import ThreadPoolExecutor
from math import comb
import boto3
from botocore.config import Config

JUDGE_MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"   # needs the us. inference-profile prefix
ARMS = {"base": "rollout_product_base.jsonl", "student": "rollout_product_student.jsonl"}
MAXCH = 24000          # cap a pathological rollout (one base trajectory ran to 70k chars)

bedrock = boto3.client("bedrock-runtime", region_name=REGION,
                       config=Config(retries={"max_attempts": 8, "mode": "adaptive"},
                                     read_timeout=300))

gold = {r["query"]: r["reward"] for r in clean_rows}            # clean_rows from step 01

def render(traj, max_obs=6):
    """Flatten one trajectory to REASONING / TOOL_CALL / OBSERVATION / RESPONSE lines."""
    out = []
    for i, s in enumerate(traj):
        c = s["completion"]; m = c["message"]
        think = (c.get("reasoning_content") or "").strip()
        if not think:                                          # serving path leaves this empty
            raw = c.get("content") or ""
            think = raw.split("</think>")[0].strip() if "</think>" in raw else ""
        if think:
            out.append(f"[step {i}] REASONING:\n{think}")
        calls = [{"name": t.get("name"), "parameters": t.get("parameters")}
                 for t in (m.get("tool_call") or [])]
        if calls:
            out.append(f"[step {i}] TOOL_CALL: {json.dumps(calls, ensure_ascii=False)}")
        for o in (m.get("obs") or []):
            res = o.get("results")
            if isinstance(res, list):
                brief = [{k: p.get(k) for k in ("product_id", "title", "price", "service")}
                         for p in res[:max_obs]]
                more = f"  (+{len(res) - max_obs} more)" if len(res) > max_obs else ""
                out.append(f"[step {i}] OBSERVATION: {json.dumps(brief, ensure_ascii=False)}{more}")
            else:
                out.append(f"[step {i}] OBSERVATION: {json.dumps(res, ensure_ascii=False)[:600]}")
        if m.get("response"):
            out.append(f"[step {i}] RESPONSE_TO_USER: {m['response']}")
    return "\n".join(out)

pairs = {}
for arm, path in ARMS.items():
    for line in open(path):
        if not line.strip():
            continue
        traj = json.loads(line)
        q = traj[0]["extra_info"]["query"]
        if q in gold:                                          # decontaminated subset only
            pairs.setdefault(q, {})[arm] = render(traj)
pairs = {q: v for q, v in pairs.items() if len(v) == 2}
print(f"{len(pairs)} clean queries with both arms rendered")

RUBRIC = """You are an expert judge of AGENTIC REASONING QUALITY for a shopping assistant.

The agent must satisfy a user's product request using tools: find_product (search),
view_product_information, recommend_product, terminate. You are shown two independent
trajectories for the SAME user request, as interleaved REASONING / TOOL_CALL /
OBSERVATION / RESPONSE_TO_USER steps.

Judge the QUALITY OF THE REASONING AND SEARCH PROCESS, not whether you agree with the
final pick. Score each trajectory 1-5 (5 best) on each axis:

- constraint_tracking: Does the reasoning extract EVERY constraint the user stated
  (product type, price bound, service/deal, size, colour, brand, attributes) and keep
  holding them as the trajectory proceeds? Dropping or inventing a constraint is bad.
- search_strategy: Are the find_product queries well-formed and targeted? When results
  are poor or empty, does it adapt (re-query, re-phrase, paginate) rather than give up
  or blindly repeat?
- evidence_grounding: Does the reasoning actually USE the returned observation data --
  citing titles/prices/services of real candidates -- rather than asserting things the
  observations do not support?
- verification: Before recommending, does it check the chosen candidate against the
  user's constraints (price in range, service offered, attributes match)?
- efficiency: Does it reach a decision without redundant, aimless or repeated steps?

Then give overall_reasoning 1-5 for each, and pick the better-reasoning trajectory.
Ignore XML/formatting issues entirely.

Reply with STRICT JSON and nothing else:
{"t1": {"constraint_tracking": int, "search_strategy": int, "evidence_grounding": int,
        "verification": int, "efficiency": int, "overall_reasoning": int},
 "t2": {...same keys...},
 "winner": "t1" | "t2" | "tie",
 "justification": "<=60 words"}"""

def clip(s):
    return s if len(s) <= MAXCH else s[:MAXCH] + "\n...[TRUNCATED]"

def judge_one(item):
    q, arms = item
    flip = random.Random(q).random() < 0.5          # seeded per query -> reproducible, unbiased order
    a1, a2 = ("base", "student") if flip else ("student", "base")
    user = (f"# User request\n{q}\n\n"
            f"# Gold specification (what a correct product must satisfy)\n"
            f"{json.dumps(gold[q], ensure_ascii=False)}\n\n"
            f"# Trajectory 1\n{clip(arms[a1])}\n\n# Trajectory 2\n{clip(arms[a2])}")
    try:
        r = bedrock.converse(modelId=JUDGE_MODEL, system=[{"text": RUBRIC}],
                             messages=[{"role": "user", "content": [{"text": user}]}],
                             inferenceConfig={"maxTokens": 1200, "temperature": 0})
        v = json.loads(re.search(r"\{.*\}", r["output"]["message"]["content"][0]["text"], re.S).group(0))
        return {"query": q, a1: v["t1"], a2: v["t2"], "first_shown": a1,
                "winner": a1 if v["winner"] == "t1" else a2 if v["winner"] == "t2" else "tie",
                "justification": v.get("justification", "")}
    except Exception as e:
        return {"query": q, "error": repr(e)[:300]}

with ThreadPoolExecutor(max_workers=8) as pool:
    judged = list(pool.map(judge_one, pairs.items()))
ok = [j for j in judged if "error" not in j]
print(f"judged {len(ok)}/{len(judged)} ({len(judged) - len(ok)} errors)")

AXES = ["constraint_tracking", "search_strategy", "evidence_grounding",
        "verification", "efficiency", "overall_reasoning"]
print(f"\n{'axis':<22}{'base':>7}{'student':>9}{'delta':>8}   sign test   W/L/T")
for ax in AXES:
    rs = [j for j in ok if ax in j["base"] and ax in j["student"]]
    b = [j["base"][ax] for j in rs]; s = [j["student"][ax] for j in rs]
    w = sum(1 for x, y in zip(b, s) if y > x); l = sum(1 for x, y in zip(b, s) if y < x)
    m, k = w + l, min(w, l)
    p = min(1.0, 2 * sum(comb(m, i) for i in range(k + 1)) / 2 ** m) if m else 1.0
    flag = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"{ax:<22}{st.mean(b):>7.2f}{st.mean(s):>9.2f}{st.mean(s)-st.mean(b):>+8.2f}   "
          f"p={p:<7.4f}{flag:<4} {w}/{l}/{len(rs)-m}")

wins = [j["winner"] for j in ok]
dec = wins.count("base") + wins.count("student")
print(f"\nhead-to-head: student {wins.count('student')} | base {wins.count('base')} | "
      f"tie {wins.count('tie')}  ->  {wins.count('student')}/{dec} = {wins.count('student')/dec:.1%}")
print(f"order balance (student shown first): {sum(1 for j in ok if j['first_shown']=='student')}/{len(ok)}")

with open("reasoning_judge_results.jsonl", "w") as f:
    for j in ok:
        f.write(json.dumps(j, ensure_ascii=False) + "\n")

# Cost: ~4k input tokens x 103 pairs on Sonnet 4.5 -- roughly $2. No endpoint, no GPU.
# To validate the judge against ground truth, join these scores to per-query ASR and check that the
# judge's preferred arm is the one that actually succeeded (measured: 19/22 = 86.4%).

### 04d. Tear down the endpoint

Delete it when done — a real-time endpoint bills for its instance while it's up.

In [ ]:
# deploy() returns an Endpoint object (no .delete_endpoint), so delete by name via the sagemaker
# client. ALWAYS tear down - an idle endpoint keeps billing (p4de/g5 are not cheap).
import boto3
sm = boto3.client("sagemaker", region_name=REGION)
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
sm.delete_model(ModelName=ENDPOINT_NAME)
print("deleted endpoint + config + model:", ENDPOINT_NAME)